# 10 · Maria Hill schedules
*Hill does not run things by hand.*

The pipeline you built by hand in chapters 03, 05 and 06 should run every morning without you. This notebook is a **job task notebook**: parameterised, idempotent and safe to retry.
You build a Lakeflow Job with three tasks (`ingest → silver → gold`) that each run this same notebook with a different `step`. Run interactively with the default `step = all`, it walks through the three steps and proves that running them twice changes nothing.

| Concept | Where you meet it |
|---|---|
| Lakeflow Jobs: a job, its tasks and their dependencies | the next section, Step 5 |
| Task parameters (the `step` widget) | Setup and Steps 1 to 3 |
| Idempotent tasks: running twice gives the same result | Steps 1 to 4 |
| Retries, repair runs, paused schedules | the next section, Step 5 |

Deep dive: [docs/10-maria-hill-schedules.md](../../docs/10-maria-hill-schedules.md)

**Needs:**
* chapter 02 (the staged files) and chapter 03 completely (`shield.bronze.incident_reports` with all five files; the checkpoint under `/Volumes/shield/landing/checkpoints/incident_reports`);
* `shield.silver.locations` from chapter 05 or 06 (the gold step joins it);
* serverless compute, and permission to create jobs.

The ingest step shares chapter 03's table and checkpoint on purpose: after chapter 03 it finds nothing new, which is the idempotency you want. Chapter 06's CHECK constraint and chapter 07's change data feed property should stay on `silver.incident_reports`, because the silver step overwrites the data and keeps the table (not verified in a workspace).
The **Expected** numbers assume the dataset from chapter 02 (10,000 incidents, seed 42, 5 % dirty) and that chapter 03 delivered all five files.

## Lakeflow Jobs and idempotent tasks

**The idea:** a **job** runs **tasks** in dependency order. Each task can get parameters, a retry policy and notifications, and the job can start on a schedule or a trigger. The tasks form a directed acyclic graph (a DAG: arrows, no loops).
Lakeflow Jobs is a Databricks feature. **Idempotency** is a pattern, not a feature: Jobs does not make your tasks idempotent. A repair run re-runs a failed task from its beginning, so a task that half-wrote its output and then appended it again duplicates data.

```
 schedule: 06:00 Europe/Berlin (created PAUSED while you learn)
        │
        ▼
 ┌────────┐     ┌────────┐     ┌────────┐     ┌──────────────┐
 │ ingest │ ──▶ │ silver │ ──▶ │  gold  │ ──▶ │ health_check │
 └────────┘     └────────┘     └────────┘     └──────────────┘
  step=ingest    step=silver    step=gold      chapter 14 notebook
```

| Term | Meaning |
|---|---|
| **Task parameter** | a key-value pair the job hands to a notebook task; the notebook reads it with `dbutils.widgets.get`. One notebook can serve many tasks and environments. |
| **Retries** | how often a failed task is started again. Good for transient errors (a cluster hiccup, a schema change Auto Loader asks you to retry, chapter 03). |
| **Repair run** | after a failure, re-run only the failed task and the tasks after it. Needs a job with two or more tasks. |
| **Schedule / trigger** | when the job starts: a cron schedule, a file arrival, a table update or continuously. |

**How this notebook implements it**

* One notebook, one widget `step` with the values `ingest`, `silver`, `gold` and `all`. The three job tasks pass `ingest`, `silver` and `gold`; you run `all` by hand.
* `ingest` copies the next staged file(s) to the raw landing zone, skipping files already delivered (the widget `deliver_files` says how many), and loads new files into `shield.bronze.incident_reports` with Auto Loader. The checkpoint skips files it has read.
* `silver` rebuilds `shield.silver.incident_reports` and `shield.silver.incident_reports_quarantine` **from bronze** with a full overwrite and the chapter 06 rules. A derived table that is fully rebuilt from its source gives the same result every time.
* `gold` runs `CREATE OR REPLACE TABLE shield.gold.daily_threat_briefing` from silver.
* Step 4 proves the point: run silver and gold twice and compare the counts.

**Why it helps:** a retry or a repair run can never double-count. The same job is defined as code in `resources/shield_job.yml` (the [deploy guide](../../docs/deploy-and-automate.md)), with a fourth task `health_check` (chapter 14) after gold.

**Not covered:** file-arrival triggers, task values (small signals between tasks), conditions between tasks, incremental `MERGE` (Try it in the doc).

## Setup

Run the widget cell first: it only creates the `step` and `deliver_files` widgets, so you can change them before the next cell reads them.

In [ ]:
dbutils.widgets.dropdown("step", "all", ["all", "ingest", "silver", "gold"], "Pipeline step ('all' = the three steps in order)")
dbutils.widgets.text("deliver_files", "1", "Files to deliver (ingest step only)")

In [ ]:
import os
import shutil
import sys
from pathlib import Path

from pyspark.sql import Window
from pyspark.sql import functions as F

sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..", "..", "src")))
from shield_data.quality import INCIDENT_RULES

step = dbutils.widgets.get("step")

STAGING, RAW = Path("/Volumes/shield/landing/staging"), Path("/Volumes/shield/landing/raw")
CP = "/Volumes/shield/landing/checkpoints"
TS = "yyyy-MM-dd'T'HH:mm:ss'Z'"


def show_checks(checks):
    """Show (check, result, expected) rows with a ✅ / ❌ for whether result matches expected."""
    fmt = lambda v: f"{v:,}" if isinstance(v, int) and not isinstance(v, bool) else str(v)
    rows = [(what, fmt(got), fmt(want), "✅" if got == want else "❌") for what, got, want in checks]
    display(spark.createDataFrame(rows, "check string, result string, expected string, ok string"))


def require(tables):
    """Fail early, with a clear message, when a table from an earlier chapter is missing."""
    missing = [t for t in tables if not spark.catalog.tableExists(t)]
    assert not missing, f"Missing tables {missing}: run the earlier chapters first (03 for bronze, 05 or 06 for silver.locations)."


print(f"step = {step}")

## 1 · The three steps

### Step 1 · Ingest
**Goal:** deliver the next staged file, if there is one, and load new files into bronze. Safe to repeat: delivered files are skipped and the checkpoint remembers what was read.

| Input | Value |
|---|---|
| `/Volumes/shield/landing/staging/incident_reports` | 5 files (10,250 rows: 10,000 incidents + 250 replayed duplicates) |
| `/Volumes/shield/landing/raw/incident_reports` | the files chapter 03 already delivered |

**Expected:** after a complete chapter 03 all five files are delivered, so this run delivers nothing and loads nothing. Bronze keeps 10,250 rows with 10,000 distinct `incident_id`s.
If you stopped chapter 03 early, each run delivers `deliver_files` more files and the counts grow; the ❌ then only means "not all files delivered yet".

In a job the task ends when the stream has finished (the Jobs service waits for it). Interactively with `step = all` the cell waits itself and shows the check table. With `step = ingest` it only starts the stream, because in a job you should not block on `awaitTermination`.

In [ ]:
def step_ingest():
    """Simulated delivery plus an incremental Auto Loader load. Idempotent: delivered files are skipped."""
    assert (STAGING / "incident_reports").is_dir(), "Run notebook 02 first (it generates the staged files)."
    feed = "incident_reports"
    (RAW / feed).mkdir(parents=True, exist_ok=True)
    have = {p.name for p in (RAW / feed).glob("*")}
    for p in [p for p in sorted((STAGING / feed).glob("*")) if p.name not in have][: int(dbutils.widgets.get("deliver_files"))]:
        shutil.copy(p, RAW / feed / p.name)
    return (
        spark.readStream.format("cloudFiles").option("cloudFiles.format", "json")
        .option("cloudFiles.schemaLocation", f"{CP}/{feed}/schema")
        .option("cloudFiles.inferColumnTypes", "true")
        .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
        .load(f"{RAW}/{feed}")
        .withColumn("_source_file", F.col("_metadata.file_path"))
        .withColumn("_ingested_at", F.current_timestamp())
        .writeStream.option("checkpointLocation", f"{CP}/{feed}/checkpoint").option("mergeSchema", "true")
        .trigger(availableNow=True).toTable("shield.bronze.incident_reports")
    )


if step in ("ingest", "all"):
    ingest_query = step_ingest()
    if step == "all":
        ingest_query.awaitTermination()
        bronze_incidents = spark.table("shield.bronze.incident_reports")
        show_checks([
            ("files in the raw landing zone", len(list((RAW / "incident_reports").glob("*"))), 5),
            ("bronze rows", bronze_incidents.count(), 10_250),
            ("bronze distinct incident_id", bronze_incidents.select("incident_id").distinct().count(), 10_000),
        ])
    print("step 'ingest' done")

### Step 2 · Silver
**Goal:** rebuild silver and quarantine from bronze with the quality rules from chapter 06. A full overwrite of a table derived from its source is idempotent.

**Expected:**

| table | rows |
|---|---|
| `shield.silver.incident_reports` | 9,500 (one per valid incident) |
| `shield.silver.incident_reports_quarantine` | 511 |

These are the numbers chapter 06 ended with. The silver step needs bronze (chapter 03).

In [ ]:
def step_silver():
    """Rebuild silver and quarantine from bronze. Full overwrite derived from bronze: same result on every run."""
    require(["shield.bronze.incident_reports"])
    parsed = spark.table("shield.bronze.incident_reports").withColumn("reported_ts", F.expr(f'try_to_timestamp(reported_at, "{TS}")'))
    failed = F.array(*[F.when(F.coalesce(F.expr(e), F.lit(False)), F.lit(None)).otherwise(F.lit(n)) for n, e in INCIDENT_RULES.items()])
    checked = parsed.withColumn("failed_rules", F.filter(failed, lambda x: x.isNotNull()))
    latest = Window.partitionBy("incident_id").orderBy(F.col("_ingested_at").desc())
    (
        checked.filter(F.size("failed_rules") == 0).drop("failed_rules")
        .withColumn("rn", F.row_number().over(latest)).filter("rn = 1").drop("rn")
        .write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("shield.silver.incident_reports")
    )
    (
        checked.filter(F.size("failed_rules") > 0).withColumn("_quarantined_at", F.current_timestamp())
        .write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("shield.silver.incident_reports_quarantine")
    )


if step in ("silver", "all"):
    step_silver()
    if step == "all":
        show_checks([
            ("silver rows", spark.table("shield.silver.incident_reports").count(), 9_500),
            ("quarantine rows", spark.table("shield.silver.incident_reports_quarantine").count(), 511),
        ])
    print("step 'silver' done")

### Step 3 · Gold
**Goal:** rebuild the daily briefing from silver.

**Expected:** `shield.gold.daily_threat_briefing` has 8,803 rows (one per day, city, incident type and severity), together 9,500 incidents and 5,681 casualties. Every incident in silver is counted once.
The gold step needs `shield.silver.incident_reports` and `shield.silver.locations` (chapter 05 or 06).

In [ ]:
def step_gold():
    """Rebuild the daily briefing from silver with CREATE OR REPLACE: same result on every run."""
    require(["shield.silver.incident_reports", "shield.silver.locations"])
    spark.sql(
        """
        CREATE OR REPLACE TABLE shield.gold.daily_threat_briefing AS
        SELECT CAST(i.reported_ts AS DATE) AS report_date, l.city, l.country, i.incident_type, i.severity,
               COUNT(*) AS incidents, SUM(i.casualties) AS casualties
        FROM shield.silver.incident_reports i JOIN shield.silver.locations l USING (location_id)
        GROUP BY ALL
        """
    )


def count_everything():
    """Row counts of the three tables this notebook writes (bronze is not rewritten by silver or gold)."""
    gold = spark.table("shield.gold.daily_threat_briefing")
    totals = gold.agg(F.sum("incidents").alias("incidents"), F.sum("casualties").alias("casualties")).first()
    return {
        "silver rows": spark.table("shield.silver.incident_reports").count(),
        "quarantine rows": spark.table("shield.silver.incident_reports_quarantine").count(),
        "gold rows": gold.count(),
        "gold incidents": totals["incidents"],
        "gold casualties": totals["casualties"],
    }


if step in ("gold", "all"):
    step_gold()
    if step == "all":
        gold_counts = count_everything()
        show_checks([
            ("gold rows", gold_counts["gold rows"], 8_803),
            ("incidents in gold", gold_counts["gold incidents"], 9_500),
            ("casualties in gold", gold_counts["gold casualties"], 5_681),
        ])
    print("step 'gold' done")

### Step 4 · Run silver and gold again
**Goal:** prove idempotency. A retry or a repair run is just "run it again", so the second run must leave every count unchanged.

**Expected:** all counts after the second run equal the counts before it (the table below lists them next to each other).
Only runs with `step = all`; a single-step job task skips this cell.

In [ ]:
if step == "all":
    counts_before = count_everything()
    step_silver()
    step_gold()
    counts_after = count_everything()
    show_checks([(name, counts_after[name], counts_before[name]) for name in counts_before])

## 2 · Make it a job

### Step 5 · Create the job and run it twice (workspace UI)
**Goal:** let the platform run the three steps in order, retry them and notify you when one fails.

1. **Jobs & Pipelines → Create → Job**. Name it `shield_daily`.
2. Task `ingest`: type *Notebook*, this notebook from your Git folder, parameter `step` = `ingest`. Compute: serverless.
3. Task `silver`: `step` = `silver`, depends on `ingest`. Task `gold`: `step` = `gold`, depends on `silver`.
4. Click **Add** next to **Retries** on each task (2 retries). Add a failure **notification** (job-level notifications are not sent when a failed task is retried; add it on each task if you want one per failed attempt). Add a schedule and start it **paused**.
5. **Run now**, then run again.

**Expected:** every task of both runs ends green (not verified in a workspace). The second run changes no counts, which the next cell checks, and shows the five newest versions of silver so you can see what wrote them.
The same job exists as code in `resources/shield_job.yml`, which also adds the `health_check` task of chapter 14 after `gold`; the deploy guide deploys it with the bundle.
The cell also works without the job: after Steps 1 to 4 it shows the same numbers.

In [ ]:
show_checks([
    ("silver rows", spark.table("shield.silver.incident_reports").count(), 9_500),
    ("quarantine rows", spark.table("shield.silver.incident_reports_quarantine").count(), 511),
    ("incidents in gold", spark.table("shield.gold.daily_threat_briefing").agg(F.sum("incidents")).first()[0], 9_500),
])
# The five newest versions of silver: with a job run you see its job id in the "job" column.
display(spark.sql("DESCRIBE HISTORY shield.silver.incident_reports").select("version", "timestamp", "userName", "operation", "job").limit(5))

## 👀 In the workspace

* **Jobs & Pipelines** → `shield_daily` → **Runs**: the matrix view shows every task of every run. Click a run, then a task, to see its output (`step 'silver' done`).
* After a failure: open the failed run and click **Repair run** to re-run only the failed task and what depends on it (a job with two or more tasks).
* **Catalog** → `shield` → `silver` → `incident_reports` → **History**: each run wrote a new version; the counts did not change.

## 🧹 Clean up

No tables are deleted: chapter 11 and the health check of chapter 14 read what this notebook rebuilt.
If you created `shield_daily` by hand, delete it in **Jobs & Pipelines** when you are done (a paused job costs nothing, but the deploy guide creates the same job from code as `shield_daily_<target>`).

## 🎯 Recap

* A job runs tasks in order with parameters, retries and a schedule; one notebook with a `step` parameter served all three tasks.
* Idempotent tasks make retries and repair runs safe: files already delivered are skipped, derived tables are rebuilt from their source, and Step 4 showed identical counts after a second run (9,500 silver, 511 quarantined, 8,803 gold rows).
* Create schedules paused while learning.

Mini-projects: [docs/10-maria-hill-schedules.md](../../docs/10-maria-hill-schedules.md#try-it-yourself)

> 💡 Next: chapter 11 puts Unity Catalog governance on these tables.